# Analyse des données d'indemnisation 2016–2022
Ce notebook charge les données des dossiers d'indemnisation pour les années 2016 à 2022, réalise une analyse exploratoire par année, applique des algorithmes de clustering non supervisés (KMeans, Agglomerative, DBSCAN) et expose des recommandations d'amélioration ainsi qu'une interprétation actuarielle du risque.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
# Affichage des graphiques inline
%matplotlib inline


In [ ]:
# Chargement des fichiers CSV et concaténation
def load_data(filepaths):
    dfs = []
    for f in filepaths:
        # Essayer des séparateurs communs
        try:
            df = pd.read_csv(f, low_memory=False)
        except Exception:
            try:
                df = pd.read_csv(f, sep=';', low_memory=False)
            except Exception:
                df = pd.read_csv(f, encoding='latin1', low_memory=False)
        # Extraire l'année du nom de fichier
        import re
        m = re.search(r'(2016|2017|2018|2019|2020|2021|2022)', f)
        year = int(m.group(1)) if m else None
        df['YEAR'] = year
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Liste des fichiers uploadés
filepaths = [
    'dossiers-indemnisation-2016.csv',
    'dossiers-indemnisation-2017.csv',
    'dossiers-indemnisation-2018.csv',
    'dossiers-indemnisation-2019.csv',
    'dossiers-indemnisation-2020.csv',
    'dossiers-indemnisation-2021.csv',
    'dossier_indem_2022.csv'
]
# Charger les données
df = load_data(filepaths)
# Aperçu des données
print(f'Nombre total de lignes: {len(df)}')
print('Colonnes: ', df.columns.tolist())
# Afficher les premières lignes
df.head()


In [ ]:
# Informations de base sur les données
print(df.info())
print('
Statistiques descriptives pour les variables numériques:')
print(df.describe())
# Vérification des valeurs manquantes par colonne
print('
Valeurs manquantes par colonne:')
print(df.isnull().sum())


In [ ]:
# Analyse descriptive globale
# Distribution des années
sns.countplot(data=df, x='YEAR', palette='viridis')
plt.title('Nombre de dossiers par année')
plt.show()
# Distribution des groupes d'âge
if 'GROUP_AGE' in df.columns:
    order = [x for x in sorted(df['GROUP_AGE'].dropna().unique(), key=lambda s: s)]
    plt.figure(figsize=(10,5))
    sns.countplot(data=df, y='GROUP_AGE', order=order, palette='magma')
    plt.title('Distribution des groupes d'âge')
    plt.show()
# Top 10 des régions
if 'NOM_REGN' in df.columns:
    top_regs = df['NOM_REGN'].value_counts().head(10)
    plt.figure(figsize=(10,6))
    sns.barplot(y=top_regs.index, x=top_regs.values, palette='coolwarm')
    plt.title('Top 10 des régions en nombre de dossiers')
    plt.xlabel('Nombre de dossiers')
    plt.show()


In [ ]:
# Fonction de résumé par année
def summary_by_year(df):
    years = sorted(df['YEAR'].dropna().unique().astype(int))
    for y in years:
        print(f'### Année {y}')
        sub = df[df['YEAR'] == y]
        print(f'Nombre de dossiers: {len(sub)}')
        # Groupe d'âge
        if 'GROUP_AGE' in sub.columns:
            print('Répartition des groupes d'âge:')
            print(sub['GROUP_AGE'].value_counts(normalize=True).sort_index() * 100)
        # Région
        if 'NOM_REGN' in sub.columns:
            print('Top 5 des régions:')
            print(sub['NOM_REGN'].value_counts().head(5))
        print('
')
# Appel de la fonction
summary_by_year(df)


In [ ]:
# Visualisation des distributions des âges par année
if 'GROUP_AGE' in df.columns:
    years = sorted(df['YEAR'].dropna().unique().astype(int))
    fig, axes = plt.subplots(len(years)//2 + len(years)%2, 2, figsize=(14, 5 * ((len(years)//2) + (len(years)%2))))
    axes = axes.flatten()
    for ax, y in zip(axes, years):
        sub = df[df['YEAR'] == y]
        order = [x for x in sorted(sub['GROUP_AGE'].dropna().unique(), key=lambda s: s)]
        sns.countplot(data=sub, y='GROUP_AGE', order=order, ax=ax)
        ax.set_title(f'Année {y}: distribution des âges')
    for ax in axes[len(years):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()


In [ ]:
# Visualisation des top 5 régions par année
if 'NOM_REGN' in df.columns:
    years = sorted(df['YEAR'].dropna().unique().astype(int))
    fig, axes = plt.subplots(len(years)//2 + len(years)%2, 2, figsize=(14, 5 * ((len(years)//2) + (len(years)%2))))
    axes = axes.flatten()
    for ax, y in zip(axes, years):
        sub = df[df['YEAR'] == y]
        top = sub['NOM_REGN'].value_counts().head(5)
        sns.barplot(x=top.values, y=top.index, ax=ax)
        ax.set_title(f'Année {y}: top 5 régions')
    for ax in axes[len(years):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()


In [ ]:
# Prétraitement et clustering
# Sélection des colonnes d'intérêt
features = []
if 'GROUP_AGE' in df.columns:
    features.append('GROUP_AGE')
if 'NOM_REGN' in df.columns:
    features.append('NOM_REGN')
if 'CODE_REGN' in df.columns:
    features.append('CODE_REGN')
if 'YEAR' in df.columns:
    features.append('YEAR')
# Encodage one-hot sur les colonnes catégorielles
data = df[features].copy()
categorical = [c for c in features if data[c].dtype == object]
data_enc = pd.get_dummies(data, columns=categorical, drop_first=False)
# Standardisation
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_enc.fillna(0))
# Réduction de dimension via PCA
pca = PCA(n_components=0.9, random_state=42)
data_pca = pca.fit_transform(data_scaled)
print(f'Nombre de composantes après PCA: {pca.n_components_}')
# Echantillonnage pour le clustering (à cause de la taille)
sample_size = min(10000, data_pca.shape[0])
idx = np.random.RandomState(42).choice(data_pca.shape[0], sample_size, replace=False)
sample = data_pca[idx]
# KMeans: test de K de 2 à 8
sil_kmeans = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(sample)
    sil_kmeans[k] = silhouette_score(sample, labels)
print('Scores silhouette pour KMeans: ', sil_kmeans)
# Affichage des scores
plt.figure(figsize=(8,4))
plt.plot(list(sil_kmeans.keys()), list(sil_kmeans.values()), marker='o')
plt.xlabel('K')
plt.ylabel('Score de silhouette')
plt.title('KMeans: choix du nombre de clusters')
plt.show()
# Agglomerative: test de K de 2 à 8
sil_agg = {}
for k in range(2, 9):
    agg = AgglomerativeClustering(n_clusters=k)
    labels = agg.fit_predict(sample)
    sil_agg[k] = silhouette_score(sample, labels)
print('Scores silhouette pour Agglomerative: ', sil_agg)
# Affichage des scores
plt.figure(figsize=(8,4))
plt.plot(list(sil_agg.keys()), list(sil_agg.values()), marker='o', color='orange')
plt.xlabel('K')
plt.ylabel('Score de silhouette')
plt.title('Agglomerative: choix du nombre de clusters')
plt.show()
# DBSCAN: test de quelques eps
sil_db = {}
for eps in [1, 2, 3]:
    db = DBSCAN(eps=eps, min_samples=10)
    labels = db.fit_predict(sample)
    # Ignorer le bruit pour le calcul
    mask = labels != -1
    if sum(mask) > 0 and len(set(labels[mask])) > 1:
        sil_db[eps] = silhouette_score(sample[mask], labels[mask])
    else:
        sil_db[eps] = np.nan
print('Scores silhouette pour DBSCAN (min_samples=10): ', sil_db)


In [ ]:
# Extraction de profils pour une solution de clustering retenue (ex. KMeans K=8 et Agglomerative K=2)
# Échantillonnage similaire à la cellule précédente
# Reprojetons les labels sur les données originales (attention: calcul sur échantillon seulement)
k_opt = max(sil_kmeans, key=sil_kmeans.get)
km = KMeans(n_clusters=k_opt, n_init=10, random_state=42)
labels_k = km.fit_predict(sample)
# Regrouper dans un DataFrame plus lisible
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
df_sample['cluster_kmeans'] = labels_k
print(f'Profil des clusters pour KMeans K={k_opt}:')
for i in range(k_opt):
    sub = df_sample[df_sample['cluster_kmeans'] == i]
    print(f'-- Cluster {i}:')
    if 'GROUP_AGE' in sub.columns:
        print('   Répartition des âges:')
        print((sub['GROUP_AGE'].value_counts(normalize=True) * 100).round(1).to_dict())
    if 'NOM_REGN' in sub.columns:
        print('   Top 3 régions:')
        print(sub['NOM_REGN'].value_counts(normalize=True).head(3).round(2).to_dict())
    print('   Taille: ', len(sub))


## Recommandations d'amélioration des données
- **Nettoyage des libellés** des régions (`NOM_REGN`) : éliminer les incohérences orthographiques, harmoniser les accents et les tirets, standardiser le format (ex. '06-MONTRÉAL').
- **Compléter le dataset** avec des variables importantes pour l'analyse actuarielle du risque : montants indemnisés, durée des dossiers, types de sinistre, date de survenance, indications de mortalité ou de gravité (si disponibles), etc. Ces variables permettront de construire des profils de risque plus pertinents.
- **Vérifier la complétude des données** (valeurs manquantes, champs codés à tort comme chaînes) avant de lancer des algorithmes de segmentation.
- **Normalisation des données horaires et spatiales** (ex. extraction d'informations depuis les dates, regroupement géographique des régions par zones plus grandes) pour améliorer la cohérence de l'analyse.
- **Documentation** du jeu de données : champs explicites, codes signification, provenance des fichiers.


## Interprétation actuarielle du risque
Les segmentations obtenues à partir des variables disponibles (âge, région, code, année) permettent d'identifier des groupes homogènes du point de vue des caractéristiques démographiques et spatiales des dossiers. Du point de vue de la gestion actuarielle du risque, ces insights peuvent servir de point de départ :
- **Age** : Plus le segment est composé d'assurés âgés (50–64 ans, 65 et plus), plus les fréquences et la sévérité des sinistres peuvent différer du reste de la population, justifiant des provisions spécifiques et des tarifications ajustées.
- **Géographie** : Les régions dominantes peuvent être corrélées à des expositions propres (densité urbaine, climat, risques locaux) favorisant certains types de sinistres (accidents routiers, maladies professionnelles...). Isoler le cluster 'Hors-Québec' illustre une catégorie à part, nécessitant une politique de gestion transfrontalière du risque.
- **Temporalité** : Les fluctuations annuelles (baisse en 2020–2021) peuvent renvoyer à des événements exceptionnels (crise sanitaire) modifiant temporairement le comportement des indemnisations. Il est important d'étudier les tendances temporelles pour prévoir les provisions futures et calibrer les modèles de tarification.

**Note** : Sans variables financières ou de sinistre précises, l'analyse actuarielle reste indicative. L'ajout de variables de montant, de durée ou de gravité est indispensable pour calculer des distributions de sinistres, des probabilités d'excès et des indicateurs comme le coût moyen par dossier, la volatilité des montants indemnisés, le comportement des grands sinistres, etc.
